<a href="https://colab.research.google.com/github/vinayachava1/RR/blob/main/Racism_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers
!pip install detoxify
!pip install tweetnlp
!pip install praw
!pip install 4chapy
!pip install tweepy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 68.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 18.0 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement 4chapy (from versions: none)
ERROR: No matching distribution found for 4chapy


In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from detoxify import Detoxify
import tweetnlp

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("IMSyPP/hate_speech_en")
tokenizer = AutoModelForSequenceClassification.from_pretrained("tomh/toxigen_roberta")
models = {
    'imsypp': {
        'model_name': 'IMSyPP/hate_speech_en',
        'labels': ['acceptable', 'inappropriate', 'offensive', 'violent']
  },
    'unitary' :{
        'model_name': 'toxic_bert',
        'labels': ['toxic', 'severe_toxic', 'threat', 'identity_hate']
    },
    'twitter_hate': {
                'model_name': 'cardiffnlp/twitter-roberta-base-hate-latest',
                'labels': ['not_hate', 'hate']
    },
    'toxigen': {
                'model_name': 'tomh/toxigen_roberta',
                'labels': ['toxic', 'not toxic']
    }
}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/865 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/790 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: tomh/toxigen_roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from google.colab import userdata

client_id = userdata.get('CLIENT_ID')
client_secret = userdata.get('CLIENT_SECRET')
user_agent = userdata.get('USER_AGENT')



In [ ]:
import praw
import pandas as pd
reddit = praw.Reddit(
    client_id=client_id,
    client_secret=client_secret,
    user_agent=user_agent,
    check_for_async=False
)
# cited : https://www.reddit.com/r/findareddit/comments/ajwgmp/whats_the_most_racist_sub_id_like_to_disclaim/
subreddits = [ 'MetaCanada', 'Conservative', 'ukpolitics', 'politics', 'immigration']

In [ ]:
reddit_posts =[]
for sub in subreddits:
  subr = reddit.subreddit(sub)

  for post in subr.top(limit = 100):
            post_data = {
            'subreddit': sub,
            'title': post.title,
            'selftext': post.selftext,
            'score': post.score,
            'id': post.id,
            'url': post.url,
            'num_comments': post.num_comments,
            'created_utc': post.created_utc,
            'author': str(post.author)
        }
  reddit_posts.append(post_data)


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

In [ ]:
import requests
import pandas as pd
import time
from html import unescape
import re

# Function to clean 4chan HTML
def clean_4chan_text(html_text):
    if not html_text:
        return ''
    text = re.sub('<.*?>', '', html_text)
    text = text.replace('&gt;', '>')
    text = unescape(text)
    return text.strip()

# Get 4chan /pol/ data
pol_posts = []

# Get catalog of threads
catalog_url = 'https://a.4cdn.org/pol/catalog.json'
response = requests.get(catalog_url)
catalog = response.json()

thread_count = 0
max_threads = 20  # Limit threads to avoid too many requests

for page in catalog:
    for thread in page['threads']:
        if thread_count >= max_threads:
            break

        thread_id = thread['no']
        print(f"Fetching thread {thread_id}...")

        # Get thread details
        thread_url = f'https://a.4cdn.org/pol/thread/{thread_id}.json'

        try:
            time.sleep(1)  # Respect rate limits
            thread_response = requests.get(thread_url)
            thread_data = thread_response.json()

            for post in thread_data['posts']:
                pol_posts.append({
                    'source': '4chan',
                    'subreddit_board': 'pol',
                    'title': post.get('sub', ''),
                    'text': clean_4chan_text(post.get('com', '')),
                    'score': None,
                    'num_comments': thread_data['posts'][0].get('replies', 0) if post['no'] == thread_id else None,
                    'created_utc': post.get('time', None),
                    'author': 'Anonymous',
                    'post_id': post['no']
                })

            thread_count += 1

        except Exception as e:
            print(f"Error fetching thread {thread_id}: {e}")
            continue

    if thread_count >= max_threads:
        break


all_posts = reddit_posts + pol_posts
df = pd.DataFrame(all_posts)

df.head()

Fetching thread 503281217...
Fetching thread 531596722...
Fetching thread 531600663...
Fetching thread 531616227...
Fetching thread 531619455...
Fetching thread 531622397...
Fetching thread 531614869...
Fetching thread 531613515...
Fetching thread 531617718...
Fetching thread 531615919...
Fetching thread 531606062...
Fetching thread 531605678...
Fetching thread 531615029...
Fetching thread 531620242...
Fetching thread 531621761...
Fetching thread 531616911...
Fetching thread 531621789...
Fetching thread 531621604...
Fetching thread 531619600...
Fetching thread 531619390...


,subreddit,title,selftext,score,id,url,num_comments,created_utc,author,source,subreddit_board,text,post_id
0,MetaCanada,"Meanwhile Trump builds a space force, Here is ...",,614.0,9ib8mn,https://i.redd.it/cadoo39bk1o11.jpg,92.0,1.537732e+09,None,NaN,NaN,NaN,NaN
1,Conservative,Epstein Didn't Kill Himself,,12812.0,1lvhfou,https://i.redd.it/ip7wu1jedubf1.jpeg,637.0,1.752064e+09,PlymouthSock,NaN,NaN,NaN,NaN
2,ukpolitics,Who’s in charge?,,3328.0,iflfst,https://i.redd.it/spnyjmdvzwi51.jpg,296.0,1.598259e+09,Prof_Black,NaN,NaN,NaN,NaN
3,politics,Trump not allowed into Scotland to escape Bide...,,96379.0,kr05j2,https://www.independent.co.uk/news/uk/home-new...,5196.0,1.609859e+09,Creddit999,NaN,NaN,NaN,NaN
4,immigration,Deportation flights from ‘Alligator Alcatraz’ ...,[Deportation flights](https://www.cnn.com/2025...,801.0,1m93i35,https://www.reddit.com/r/immigration/comments/...,302.0,1.753460e+09,cnn,NaN,NaN,NaN,NaN


In [ ]:
from google.colab import userdata
bearer_token = userdata.get('BEARER_TOKEN')

In [ ]:
import requests

url = "https://data.gopher-ai.com/api/v1/search/live"
headers = {
    "Authorization": f"Bearer {bearer_token}",
    "Content-Type": "application/json"
}

payload = {
    "type": "twitter",
    "arguments": {
        "type": "searchbyquery",
        "query": "immigration", # Placeholder query, user can specify topics later
        "max_results": 5
    }
}

try:
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status() # Raise an exception for HTTP errors
    twitter_data = response.json()
    print(twitter_data)
except requests.exceptions.RequestException as e:
    print(f"Error making API call: {e}")
    if response.status_code:
        print(f"Status Code: {response.status_code}")
        print(f"Response: {response.text}")

Error making API call: 403 Client Error: Forbidden for url: https://data.gopher-ai.com/api/v1/search/live
Status Code: 403
Response: {"error":"Insufficient quota","details":"This request requires 15 credits, but you only have 5 credits available."}


In [ ]:
import requests

# The UUID was returned from the previous successful API call for submitting the search.
# It's stored in `twitter_data` from the execution of cell ABd6-HMswwV7.
if 'twitter_data' in locals() and 'uuid' in twitter_data:
    result_uuid = twitter_data['uuid']
else:
    print("Error: UUID not found. Please run the previous cell to submit a search request first.")
    result_uuid = None

if result_uuid:
    result_url = f"https://data.gopher-ai.com/api/v1/search/live/result/{result_uuid}"
    headers = {
        "Authorization": f"Bearer {bearer_token}"
    }

    try:
        response = requests.get(result_url, headers=headers)
        response.raise_for_status() # Raise an exception for HTTP errors
        gopher_ai_results = response.json()
        print(gopher_ai_results)
    except requests.exceptions.RequestException as e:
        print(f"Error making API call to retrieve results: {e}")
        if response.status_code:
            print(f"Status Code: {response.status_code}")
            print(f"Response: {response.text}")

Error: UUID not found. Please run the previous cell to submit a search request first.
